In [1]:
from pathlib import Path
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu

In [ ]:
TILE_IDS = [
    '515000_3530000',
    '515000_3531000',
    '511000_3528000',
]
YEAR = '2022'
SITE_ID = 'SRER'
SITE_NAME = 'Santa_Rita_Experimental_Range_NEON'

OUTPUT_DIR = Path.cwd() / "results" / SITE_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SITE_PATH = Path(f'/projectnb/modislc/users/fache/data/NEON/{SITE_NAME}')

# tiles are 1000mx1000m
# read at 1000x1000 so 1mx1m, size of vi and lidar, rgb is 0.1m
OUT_SHAPE = (1000, 1000)

RGB_SHAPE = (10000, 10000)   # native 10cm
SAVI_SHAPE = (1000, 1000)    # native 1m
UPSAMPLE_FACTOR = 10         # 1m -> 10cm

OVERLAY_ALPHA = 0.45
OVERLAY_COLOR = (1.0, 0.0, 0.0)

In [ ]:
def build_paths(tile_id):
    rgb = SITE_PATH / 'NEON_images-camera-ortho-mosaic' / f'NEON.D14.{SITE_ID}.DP3.30010.001.2022-08.basic' / f'{YEAR}_{SITE_ID}_5_{tile_id}_image.tif'
    vi_dir = SITE_PATH / 'NEON_indices-veg-spectrometer-bidir-mosaic' / f'NEON.D14.{SITE_ID}.DP3.30026.002.2022-08.basic' / f'NEON_D14_{SITE_ID}_DP3_{tile_id}_bidirectional_VegIndices'
    ndvi = vi_dir / f'NEON_D14_{SITE_ID}_DP3_{tile_id}_bidirectional_NDVI.tif'
    savi = vi_dir / f'NEON_D14_{SITE_ID}_DP3_{tile_id}_bidirectional_SAVI.tif'
    return rgb, ndvi, savi

In [ ]:
def read_raster(path, out_shape=None, bands=None):
    """Read raster. Returns array and native shape."""
    with rasterio.open(path) as src:
        native_shape = (src.height, src.width)
        native_band_count = src.count
        nodata = src.nodata

        if bands is None:
            bands = list(range(1, native_band_count + 1))

        if out_shape is None:
            arr = src.read(bands)
        else:
            arr = src.read(bands, out_shape=(len(bands), out_shape[0], out_shape[1]))

    arr = arr.astype(np.float32)
    if nodata is not None:
        arr[arr == nodata] = np.nan
    return arr, native_shape, native_band_count

def valid(arr):
    """Return finite values only, flattened."""
    v = arr.ravel()
    return v[np.isfinite(v)]


def summary_stats(values, label):
    v = values[np.isfinite(values)]
    if v.size == 0:
        print(f"{label}: all NaN / no valid pixels")
        return
    pcts = np.percentile(v, [1, 5, 25, 50, 75, 95, 99])
    print(f"{label}:")
    print(f"n valid : {v.size:,}")
    print(f"min     : {v.min():.4f}")
    print(f"max     : {v.max():.4f}")
    print(f"mean    : {v.mean():.4f}")
    print(f"median  : {np.median(v):.4f}")
    print(f"p1      : {pcts[0]:.4f}")
    print(f"p5      : {pcts[1]:.4f}")
    print(f"p25     : {pcts[2]:.4f}")
    print(f"p75     : {pcts[4]:.4f}")
    print(f"p95     : {pcts[5]:.4f}")
    print(f"p99     : {pcts[6]:.4f}")

def pool_across_tiles(tile_ids):
    """
    Pool R, G, B, NDVI, SAVI across all tiles.

    Inputs:
        tile_ids : list of tile ID strings
    Returns:
        dict with keys R, G, B, NDVI, SAVI mapped to concatenated 1D arrays.
    """
    r_pool, g_pool, b_pool, ndvi_pool, savi_pool = [], [], [], [], []

    for tid in tile_ids:
        rgb_path, ndvi_path, savi_path = build_paths(tid)
        print(f"\nLoading tile {tid}")

        rgb, rgb_native, rgb_bands = read_raster(rgb_path, out_shape=OUT_SHAPE, bands=[1, 2, 3])
        print(f"RGB native {rgb_native} ({rgb_bands} bands)  read {rgb.shape[1:]}")
        r_pool.append(valid(rgb[0]))
        g_pool.append(valid(rgb[1]))
        b_pool.append(valid(rgb[2]))

        ndvi, ndvi_native, ndvi_bands = read_raster(ndvi_path, out_shape=OUT_SHAPE, bands=[1])
        ndvi = ndvi[0]
        print(f"NDVI native {ndvi_native} ({ndvi_bands} bands)  read {ndvi.shape}")
        ndvi_pool.append(valid(ndvi))

        savi, savi_native, savi_bands = read_raster(savi_path, out_shape=OUT_SHAPE, bands=[1])
        savi = savi[0]
        print(f"SAVI native {savi_native} ({savi_bands} bands)  read {savi.shape}")
        savi_pool.append(valid(savi))

    return {
        "R":    np.concatenate(r_pool),
        "G":    np.concatenate(g_pool),
        "B":    np.concatenate(b_pool),
        "NDVI": np.concatenate(ndvi_pool),
        "SAVI": np.concatenate(savi_pool),
    }

def candidate_thresholds(savi_values):
    """
    Compute candidate SAVI bare thresholds.

    Inputs:
        savi_values : 1D array of pooled SAVI values
    Returns:
        dict of {name: threshold value}
    """
    return {
        "otsu":            float(threshold_otsu(savi_values)),
        "p15":             float(np.percentile(savi_values, 15)),
        "p25":             float(np.percentile(savi_values, 25))
    }


def bare_fraction(savi_values, threshold):
    """Fraction of pixels below the given threshold."""
    return float((savi_values < threshold).sum() / savi_values.size)

In [ ]:
pooled = pool_across_tiles(TILE_IDS)

print(f"\n=== Pooled counts across {len(TILE_IDS)} tiles ===")
for k, v in pooled.items():
    print(f"  {k}: {v.size:,} valid pixels")

# print("\n=== Pooled summary statistics ===")
# print("\nRGB (DN 0-255):")
# summary_stats(pooled["R"], "R")
# summary_stats(pooled["G"], "G")
# summary_stats(pooled["B"], "B")
# print("\nNDVI:")
# summary_stats(pooled["NDVI"], "NDVI")
# print("\nSAVI:")
# summary_stats(pooled["SAVI"], "SAVI")

thresholds = candidate_thresholds(pooled["SAVI"])
print("\n=== SAVI candidate bare thresholds ===")
for name, t in thresholds.items():
    frac = bare_fraction(pooled["SAVI"], t)
    print(f"{name:15s} = {t:.4f} -> bare fraction = {frac*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 12))

# (1) RGB overlaid
ax = axes[0]
rgb_bins = np.linspace(0, 255, 256)
for name, color in zip(["R", "G", "B"], ["red", "green", "blue"]):
    ax.hist(pooled[name], bins=rgb_bins, alpha=0.5, color=color, label=name)
ax.set_title(f"Pooled RGB Histogram for Tiles ({', '.join(TILE_IDS)})")
ax.set_xlabel("Digital Number (0-255)")
ax.set_ylabel("Pixel count")
ax.legend()

# (2) NDVI
ax = axes[1]
ax.hist(pooled["NDVI"], bins=256, color="darkgreen", alpha=0.75)
ax.set_title(f"Pooled NDVI Histogram for Tiles ({', '.join(TILE_IDS)})")
ax.set_xlabel("NDVI")
ax.set_ylabel("Pixel count")

# (3) SAVI with candidate thresholds
ax = axes[2]
ax.hist(pooled["SAVI"], bins=256, color="darkolivegreen", alpha=0.75)
line_styles = {
    "otsu":            {"color": "red",    "ls": "-",  "lw": 1.5},
    "p15":             {"color": "orange", "ls": "--", "lw": 1.2},
    "p25":             {"color": "gold",   "ls": "--", "lw": 1.2}
}
for name, t in thresholds.items():
    s = line_styles[name]
    ax.axvline(t, color=s["color"], linestyle=s["ls"], linewidth=s["lw"], label=f"{name} = {t:.3f}")
ax.set_title(f"Pooled SAVI Histogram for Tiles ({', '.join(TILE_IDS)})")
ax.set_xlabel("SAVI")
ax.set_ylabel("Pixel count")
ax.legend(loc="upper right", fontsize=8)

fig.tight_layout()
out_path = OUTPUT_DIR / f"pooled_histograms_{YEAR}.png"
fig.savefig(out_path, dpi=150)
plt.close(fig)
print(f"\nFigure saved: {out_path}")

In [ ]:
def upsample_mask_nn(mask, factor):
    """
    Nearest-neighbor upsample of a 2D boolean mask by an integer factor.

    Inputs:
        mask   : 2D boolean or uint8 array
        factor : integer upsample factor
    Returns:
        Upsampled 2D array with shape (H*factor, W*factor)
    """
    return np.repeat(np.repeat(mask, factor, axis=0), factor, axis=1)


def rgb_to_display(rgb):
    """
    Convert a (3, H, W) uint8-scale RGB array to (H, W, 3) float in [0, 1] for imshow.
    NaNs (from nodata) are set to 0.
    """
    arr = np.transpose(rgb, (1, 2, 0))
    arr = np.nan_to_num(arr, nan=0.0)
    arr = np.clip(arr / 255.0, 0.0, 1.0)
    return arr


def overlay_bare_on_rgb(ax, rgb_disp, bare_mask_10cm, title):
    """Show RGB and overlay bare-mask pixels as translucent red."""
    ax.imshow(rgb_disp, interpolation='nearest')
    overlay = np.zeros((*bare_mask_10cm.shape, 4), dtype=np.float32)
    overlay[..., 0] = OVERLAY_COLOR[0]
    overlay[..., 1] = OVERLAY_COLOR[1]
    overlay[..., 2] = OVERLAY_COLOR[2]
    overlay[..., 3] = np.where(bare_mask_10cm, OVERLAY_ALPHA, 0.0)
    ax.imshow(overlay, interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])


def process_tile(tile_id, thresholds):
    """
    Build a 1x3 overlay figure for one tile, one panel per candidate threshold.

    Inputs:
        tile_id    : tile ID string
        thresholds : dict of {name: threshold value}
    Returns:
        Path to saved figure
    """
    rgb_path, _, savi_path = build_paths(tile_id)

    print(f"\nTile {tile_id}")
    rgb, rgb_native, rgb_bands = read_raster(rgb_path, out_shape=RGB_SHAPE, bands=[1, 2, 3])
    print(f"  RGB  native {rgb_native} ({rgb_bands} bands)  read {rgb.shape[1:]}")

    savi, savi_native, savi_bands = read_raster(savi_path, out_shape=SAVI_SHAPE, bands=[1])
    savi = savi[0]
    print(f"  SAVI native {savi_native} ({savi_bands} bands)  read {savi.shape}")

    rgb_disp = rgb_to_display(rgb)
    del rgb  # free memory

    panel_names = ["otsu", "p15", "p25"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, name in zip(axes, panel_names):
        t = thresholds[name]
        bare_1m = (savi < t)
        n_valid = np.isfinite(savi).sum()
        frac = float(bare_1m.sum() / n_valid) if n_valid > 0 else float('nan')
        bare_10cm = upsample_mask_nn(bare_1m, UPSAMPLE_FACTOR)
        title = f"{name} = {t:.3f}  |  bare = {frac*100:.1f}%"
        overlay_bare_on_rgb(ax, rgb_disp, bare_10cm, title)
        del bare_10cm

    fig.suptitle(f"Bare mask overlay | Tile {tile_id} | {SITE_ID} {YEAR}", fontsize=12)
    fig.tight_layout()
    out_path = OUTPUT_DIR / f"bare_overlay_{tile_id}_{YEAR}.png"
    fig.savefig(out_path, dpi=120)
    plt.close(fig)
    print(f"Figure saved: {out_path}")
    return out_path

In [ ]:
# print("=== Pooled candidate thresholds ===")
# for name, t in thresholds.items():
#     print(f"  {name:5s} = {t:.4f}")

# for tid in TILE_IDS:
#     process_tile(tid, thresholds)

In [ ]:
SAVI_BARE_THRESHOLD = 0.2 # NOTE determiend by viewing histograms and overlays

In [ ]:
"""
Phase 2 Step 6: Per-tile bare mask + confidence from SAVI.

Given the finalized bare threshold (0.2), compute a boolean bare mask and a
per-pixel confidence layer for each tile, and write both as GeoTIFFs
preserving CRS and transform from the source SAVI raster.
"""

In [ ]:
def compute_bare_mask(savi_path, threshold):
    """
    Compute bare mask and confidence layer from a SAVI raster.

    Purpose:
        Identify pixels below a SAVI threshold as bare-soil candidates and
        assign a per-pixel confidence proportional to distance below the
        threshold. Preserves geospatial metadata for downstream writing.

    Inputs:
        savi_path : pathlib.Path
            Path to a single-band SAVI GeoTIFF (native 1m).
        threshold : float
            SAVI value below which a pixel is flagged bare. Default 0.2.

    Outputs:
        mask       : np.ndarray of uint8, shape (H, W)
                     1 = bare, 0 = not bare, 255 = nodata.
        confidence : np.ndarray of float32, shape (H, W)
                     Distance-below-threshold scaled to [0, 1] for bare
                     pixels; 0 for non-bare pixels; NaN for nodata.
                     confidence = clip((threshold - savi) / threshold, 0, 1)
        profile    : dict
                     Rasterio profile from the source SAVI raster, useful
                     for writing outputs with matching CRS and transform.
    """
    with rasterio.open(savi_path) as src:
        savi = src.read(1).astype(np.float32)
        profile = src.profile.copy()
        nodata = src.nodata

    valid = np.isfinite(savi)
    if nodata is not None:
        valid &= (savi != nodata)

    # Boolean bare mask
    bare = np.zeros(savi.shape, dtype=bool)
    bare[valid] = savi[valid] < threshold

    # uint8 mask with nodata marker
    mask = np.full(savi.shape, 255, dtype=np.uint8)
    mask[valid & bare] = 1
    mask[valid & ~bare] = 0

    # Confidence: (threshold - savi) / threshold, clipped to [0, 1]
    confidence = np.full(savi.shape, np.nan, dtype=np.float32)
    conf_valid = np.clip((threshold - savi[valid]) / threshold, 0.0, 1.0)
    confidence[valid] = conf_valid

    return mask, confidence, profile


def write_geotiff(array, profile, out_path, dtype, nodata):
    """Write a single-band GeoTIFF matching the source profile."""
    prof = profile.copy()
    prof.update(count=1, dtype=dtype, nodata=nodata, compress='lzw')
    with rasterio.open(out_path, 'w', **prof) as dst:
        dst.write(array, 1)


def process_tile(tile_id, threshold):
    """Compute and write bare mask + confidence for one tile."""
    _, _, savi_path = build_paths(tile_id)

    print(f"\nTile {tile_id}")
    print(f"SAVI: {savi_path.name}")

    mask, confidence, profile = compute_bare_mask(savi_path, threshold=threshold)

    valid_pixels = (mask != 255).sum()
    bare_pixels = (mask == 1).sum()
    bare_frac = bare_pixels / valid_pixels if valid_pixels > 0 else float('nan')
    print(f"valid pixels : {valid_pixels:,}")
    print(f"bare pixels  : {bare_pixels:,}")
    print(f"bare fraction: {bare_frac*100:.2f}%  @ threshold {threshold}")

    mask_path = OUTPUT_DIR / f"bare_mask_{tile_id}_{YEAR}.tif"
    conf_path = OUTPUT_DIR / f"bare_confidence_{tile_id}_{YEAR}.tif"

    write_geotiff(mask, profile, mask_path, dtype='uint8', nodata=255)
    write_geotiff(confidence, profile, conf_path, dtype='float32', nodata=np.nan)

    print(f"wrote {mask_path.name}")
    print(f"wrote {conf_path.name}")

In [ ]:
print(f"Bare threshold: SAVI < {SAVI_BARE_THRESHOLD}")
for tid in TILE_IDS:
    process_tile(tid, threshold=SAVI_BARE_THRESHOLD)